In [1]:
import config
import torch
import sys
sys.path.append('../util')
sys.path.append('../others_networks')
sys.path.append('../UMergeNet')

#Clone the repository: #https://github.com/LeeJunHyun/Image_Segmentation/blob/master/network.py
#Add the clone location to be able to import from UNets
sys.path.append("/media/calculon/TUDAO/Image_Segmentation")
from network import U_Net
from network import AttU_Net
from network import R2AttU_Net
from util import measure_glops_fps, clear_gpu

from GenericDatasetReader import *

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

result_path          = './colab_unets/'
models_stats         = {}

In [2]:
in_channels     = config.in_channels
output_channels = 1
num_classes     = config.num_classes

train_loader, test_loader, val_loader = get_datasets(dataset_dir=config.dataset_path, 
                                                    resolution=config.dataset_resolution, 
                                                    batch_size=config.batch_size)

In [4]:
model = U_Net(img_ch=in_channels, output_ch=output_channels)
models_stats['unet'] = measure_glops_fps(model, val_loader, model_filename=f"{result_path}/UNet-1-epochs300.pth")
print(models_stats['unet'])

[INFO] Register zero_ops() for <class 'torch.nn.modules.pooling.MaxPool2d'>.
[INFO] Register count_convNd() for <class 'torch.nn.modules.conv.Conv2d'>.
[INFO] Register count_normalization() for <class 'torch.nn.modules.batchnorm.BatchNorm2d'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.activation.ReLU'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.container.Sequential'>.
[INFO] Register count_upsample() for <class 'torch.nn.modules.upsampling.Upsample'>.
{'params': 34527041.0, 'gflops': 50.165563392, 'gpu_fps': 184.12098492648877, 'cpu_fps': 3.4606858116786063}


In [5]:
model = AttU_Net(img_ch=in_channels, output_ch=output_channels)
models_stats['att_unet'] = measure_glops_fps(model, val_loader, model_filename=f"{result_path}/AttU_Net-1-epochs300.pth")
print(models_stats['att_unet'])

[INFO] Register zero_ops() for <class 'torch.nn.modules.pooling.MaxPool2d'>.
[INFO] Register count_convNd() for <class 'torch.nn.modules.conv.Conv2d'>.
[INFO] Register count_normalization() for <class 'torch.nn.modules.batchnorm.BatchNorm2d'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.activation.ReLU'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.container.Sequential'>.
[INFO] Register count_upsample() for <class 'torch.nn.modules.upsampling.Upsample'>.
{'params': 34878573.0, 'gflops': 51.015008576, 'gpu_fps': 168.1691990705456, 'cpu_fps': 3.0261405616627197}


In [6]:
model = R2AttU_Net(img_ch=in_channels, output_ch=output_channels)
models_stats['r2att_unet'] = measure_glops_fps(model, val_loader, model_filename=f"{result_path}/R2AttU_Net-1-epochs300.pth")
print(models_stats['r2att_unet'])

[INFO] Register zero_ops() for <class 'torch.nn.modules.pooling.MaxPool2d'>.
[INFO] Register count_upsample() for <class 'torch.nn.modules.upsampling.Upsample'>.
[INFO] Register count_convNd() for <class 'torch.nn.modules.conv.Conv2d'>.
[INFO] Register count_normalization() for <class 'torch.nn.modules.batchnorm.BatchNorm2d'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.activation.ReLU'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.container.Sequential'>.
{'params': 39442925.0, 'gflops': 117.928116544, 'gpu_fps': 67.80187441661845, 'cpu_fps': 1.3083126709555057}


In [8]:
from ULite import ULite
model = ULite(in_channels=in_channels, out_channels=output_channels)
models_stats['ulite'] = measure_glops_fps(model, val_loader, model_filename=f"ULite/ULite-1-epochs300.pth")
print(models_stats['ulite'])

[INFO] Register count_convNd() for <class 'torch.nn.modules.conv.Conv2d'>.
[INFO] Register count_normalization() for <class 'torch.nn.modules.batchnorm.BatchNorm2d'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.pooling.MaxPool2d'>.
[INFO] Register count_upsample() for <class 'torch.nn.modules.upsampling.Upsample'>.
{'params': 878417.0, 'gflops': 0.579545344, 'gpu_fps': 1256.3448078065057, 'cpu_fps': 27.963868436951593}


In [9]:
from UNext import UNext
model = UNext(input_channels=in_channels, num_classes=output_channels)
models_stats['unext'] = measure_glops_fps(model, val_loader, model_filename=f"UNext/UNext-1-epochs300.pth")
print(models_stats['unext'])

[INFO] Register count_convNd() for <class 'torch.nn.modules.conv.Conv2d'>.
[INFO] Register count_normalization() for <class 'torch.nn.modules.batchnorm.BatchNorm2d'>.
[INFO] Register count_normalization() for <class 'torch.nn.modules.normalization.LayerNorm'>.
[INFO] Register count_linear() for <class 'torch.nn.modules.linear.Linear'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.dropout.Dropout'>.
[INFO] Register count_softmax() for <class 'torch.nn.modules.activation.Softmax'>.


/home/calculon/miniconda3/envs/pytorch/lib/python3.10/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


{'params': 1471921.0, 'gflops': 0.438720128, 'gpu_fps': 3529.739996139364, 'cpu_fps': 121.68564911516843}


In [10]:
from DeepLabV3 import DeepLabV3MobilenetV2Wrapper
model = DeepLabV3MobilenetV2Wrapper(in_channels=in_channels, out_channels=output_channels)
models_stats['deeplabv3'] = measure_glops_fps(model, val_loader, model_filename=f"DeepLabV3/DeepLabV3MobilenetV2Wrapper-1-epochs300.pth")
print(models_stats['deeplabv3'])

[INFO] Register count_convNd() for <class 'torch.nn.modules.conv.Conv2d'>.
[INFO] Register count_normalization() for <class 'torch.nn.modules.batchnorm.BatchNorm2d'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.activation.ReLU6'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.container.Sequential'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.activation.ReLU'>.
[INFO] Register count_adap_avgpool() for <class 'torch.nn.modules.pooling.AdaptiveAvgPool2d'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.dropout.Dropout'>.
{'params': 12647937.0, 'gflops': 0.82144768, 'gpu_fps': 3184.5083455366303, 'cpu_fps': 68.17525243581825}


In [11]:
from UMergeNet import UMergeNet, ConvType

In [12]:
model = UMergeNet(in_channels=in_channels, out_channels=output_channels, conv_type=ConvType.Axial)
models_stats['umergenet_axial'] = measure_glops_fps(model, val_loader, model_filename=f"UMergeNet/UMergeNet-axial-1-epochs300.pth")
print(models_stats['umergenet_axial'])

[INFO] Register zero_ops() for <class 'torch.nn.modules.pooling.MaxPool2d'>.
[INFO] Register count_upsample() for <class 'torch.nn.modules.upsampling.Upsample'>.
[INFO] Register count_convNd() for <class 'torch.nn.modules.conv.Conv2d'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.container.Sequential'>.
[INFO] Register count_normalization() for <class 'torch.nn.modules.batchnorm.BatchNorm2d'>.
{'params': 1726801.0, 'gflops': 0.882545664, 'gpu_fps': 1233.6458095314954, 'cpu_fps': 34.179738002041674}


In [13]:
model = UMergeNet(in_channels=in_channels, out_channels=output_channels, conv_type=ConvType.Axial, merger_groups='dw', encoder_groups='dw', decoder_groups='dw')
models_stats['umergenet_axial_dw'] = measure_glops_fps(model, val_loader, model_filename=f"UMergeNet/UMergeNet-axial-dw-1-epochs300.pth")
print(models_stats['umergenet_axial_dw'])

[INFO] Register zero_ops() for <class 'torch.nn.modules.pooling.MaxPool2d'>.
[INFO] Register count_upsample() for <class 'torch.nn.modules.upsampling.Upsample'>.
[INFO] Register count_convNd() for <class 'torch.nn.modules.conv.Conv2d'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.container.Sequential'>.
[INFO] Register count_normalization() for <class 'torch.nn.modules.batchnorm.BatchNorm2d'>.
{'params': 685497.0, 'gflops': 0.480886784, 'gpu_fps': 1412.4283592465322, 'cpu_fps': 34.80447148616541}


In [14]:
model = UMergeNet(in_channels=in_channels, out_channels=output_channels, conv_type=ConvType.Atrous)
models_stats['umergenet_atrous'] = measure_glops_fps(model, val_loader, model_filename=f"UMergeNet/UMergeNet-atrous-1-epochs300.pth")
print(models_stats['umergenet_atrous'])

[INFO] Register zero_ops() for <class 'torch.nn.modules.pooling.MaxPool2d'>.
[INFO] Register count_upsample() for <class 'torch.nn.modules.upsampling.Upsample'>.
[INFO] Register count_convNd() for <class 'torch.nn.modules.conv.Conv2d'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.container.Sequential'>.
[INFO] Register count_normalization() for <class 'torch.nn.modules.batchnorm.BatchNorm2d'>.
{'params': 2487385.0, 'gflops': 1.187214336, 'gpu_fps': 1223.2867725481124, 'cpu_fps': 33.59733697522416}


In [15]:
model = UMergeNet(in_channels=in_channels, out_channels=output_channels, conv_type=ConvType.Atrous, merger_groups='dw', encoder_groups='dw', decoder_groups='dw')
models_stats['umergenet_atrous_dw'] = measure_glops_fps(model, val_loader, model_filename=f"UMergeNet/UMergeNet-atrous-dw-1-epochs300.pth")
print(models_stats['umergenet_atrous_dw'])

[INFO] Register zero_ops() for <class 'torch.nn.modules.pooling.MaxPool2d'>.
[INFO] Register count_upsample() for <class 'torch.nn.modules.upsampling.Upsample'>.
[INFO] Register count_convNd() for <class 'torch.nn.modules.conv.Conv2d'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.container.Sequential'>.
[INFO] Register count_normalization() for <class 'torch.nn.modules.batchnorm.BatchNorm2d'>.
{'params': 701513.0, 'gflops': 0.512899072, 'gpu_fps': 1680.9540746856203, 'cpu_fps': 39.888010923710766}


In [16]:
model = UMergeNet(in_channels=in_channels, out_channels=output_channels, conv_type=ConvType.Standard)
models_stats['umergenet_standard'] = measure_glops_fps(model, val_loader, model_filename=f"UMergeNet/UMergeNet-normal-1-epochs300.pth")
print(models_stats['umergenet_standard'])

[INFO] Register zero_ops() for <class 'torch.nn.modules.pooling.MaxPool2d'>.
[INFO] Register count_upsample() for <class 'torch.nn.modules.upsampling.Upsample'>.
[INFO] Register count_convNd() for <class 'torch.nn.modules.conv.Conv2d'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.container.Sequential'>.
[INFO] Register count_normalization() for <class 'torch.nn.modules.batchnorm.BatchNorm2d'>.
{'params': 4150297.0, 'gflops': 1.851945984, 'gpu_fps': 956.5308315730359, 'cpu_fps': 32.071631239140636}


In [17]:
model = UMergeNet(in_channels=in_channels, out_channels=output_channels, conv_type=ConvType.Standard, merger_groups='dw', encoder_groups='dw', decoder_groups='dw')
models_stats['umergenet_standard_dw'] = measure_glops_fps(model, val_loader, model_filename=f"UMergeNet/UMergeNet-normal-dw-1-epochs300.pth")
print(models_stats['umergenet_standard_dw'])

[INFO] Register zero_ops() for <class 'torch.nn.modules.pooling.MaxPool2d'>.
[INFO] Register count_upsample() for <class 'torch.nn.modules.upsampling.Upsample'>.
[INFO] Register count_convNd() for <class 'torch.nn.modules.conv.Conv2d'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.container.Sequential'>.
[INFO] Register count_normalization() for <class 'torch.nn.modules.batchnorm.BatchNorm2d'>.
{'params': 739913.0, 'gflops': 0.582744064, 'gpu_fps': 1490.51910744135, 'cpu_fps': 39.83009053970253}


In [3]:
from ultralytics import YOLO 

model = YOLO(f"yolo11/train/weights/best.pt")
models_stats['yolov11'] = measure_glops_fps(model, val_loader)
print(models_stats['yolov11'])

YOLO11n-seg summary: 203 layers, 2,842,803 parameters, 0 gradients, 9.7 GFLOPs
{'params': 0, 'gflops': 0, 'gpu_fps': 781.7949481747733, 'cpu_fps': 602.3714322285213}
